# Oscar Model Dataset

This notebook generates a standalone model-ready dataset from `oscars.csv` only. It does not merge with `outputs/eligible_movie_features.csv`.

The output is a row-level Oscar table with a binary winner target and a set of numeric features derived from ceremony metadata, nomination structure, and the award category labels.

In [4]:
import os
import re
from pathlib import Path

import pandas as pd

base = Path('.')
oscars = pd.read_csv(base / 'oscars.csv', sep='\t', dtype=str)


def parse_award_year(value):
    if pd.isna(value) or not str(value).strip():
        return pd.NA, pd.NA

    text = str(value).strip()
    match = re.match(r'^(\d{4})(?:/(\d{2,4}))?$', text)
    if not match:
        return pd.NA, pd.NA

    start_year = int(match.group(1))
    suffix = match.group(2)
    if suffix is None:
        end_year = start_year
    elif len(suffix) == 2:
        end_year = int(str(start_year)[:2] + suffix)
    else:
        end_year = int(suffix)
    return start_year, end_year


def count_pipe_items(value):
    if pd.isna(value) or not str(value).strip():
        return 0
    return sum(1 for item in str(value).split('|') if item.strip())


model_df = oscars.copy()
model_df['target_winner'] = model_df['Winner'].fillna('').astype(str).str.lower().isin({'true', '1', 'yes', 'y'}).astype(int)
model_df[['award_year_start', 'award_year_end']] = model_df['Year'].apply(lambda value: pd.Series(parse_award_year(value)))
model_df['award_year_span'] = model_df['award_year_end'] - model_df['award_year_start']
model_df['ceremony_num'] = pd.to_numeric(model_df['Ceremony'], errors='coerce').fillna(0).astype(int)
model_df['has_film'] = model_df['Film'].fillna('').str.strip().ne('').astype(int)
model_df['film_count'] = model_df['Film'].apply(count_pipe_items)
model_df['has_nominees'] = model_df['Nominees'].fillna('').str.strip().ne('').astype(int)
model_df['nominee_id_count'] = model_df['NomineeIds'].apply(count_pipe_items)
model_df['has_detail'] = model_df['Detail'].fillna('').str.strip().ne('').astype(int)
model_df['has_note'] = model_df['Note'].fillna('').str.strip().ne('').astype(int)

class_clean = model_df['Class'].fillna('Unknown').astype(str).str.strip().replace('', 'Unknown')
class_dummies = pd.get_dummies(class_clean, prefix='class').astype(int)

canonical_clean = model_df['CanonicalCategory'].fillna('Unknown').astype(str).str.strip().replace('', 'Unknown')
top_canonical = canonical_clean.value_counts().head(20).index
canonical_bucketed = canonical_clean.where(canonical_clean.isin(top_canonical), 'Other')
canonical_dummies = pd.get_dummies(canonical_bucketed, prefix='canon').astype(int)

output_columns = [
    'ceremony_num',
    'award_year_start',
    'award_year_end',
    'award_year_span',
    'has_film',
    'film_count',
    'has_nominees',
    'nominee_id_count',
    'has_detail',
    'has_note',
    'target_winner',
]

model_ready = pd.concat([model_df[output_columns], class_dummies, canonical_dummies], axis=1)
model_ready = model_ready.fillna(0)
model_ready.columns = [re.sub(r'[^A-Za-z0-9_]+', '_', col) for col in model_ready.columns]
model_ready = model_ready.astype(int)

output_path = base / 'outputs' / 'oscars_model_dataset.csv'
os.makedirs(output_path.parent, exist_ok=True)
model_ready.to_csv(output_path, index=False)

print(f'Model-ready Oscar dataset saved to {output_path}')
print(f'Rows: {len(model_ready):,}')
print(f'Columns: {len(model_ready.columns):,}')
print(f'Target win rate: {model_ready["target_winner"].mean():.2%}')
print(model_ready.head(3).to_string(index=False))

Model-ready Oscar dataset saved to outputs/oscars_model_dataset.csv
Rows: 12,137
Columns: 40
Target win rate: 28.96%
 ceremony_num  award_year_start  award_year_end  award_year_span  has_film  film_count  has_nominees  nominee_id_count  has_detail  has_note  target_winner  class_Acting  class_Directing  class_Music  class_Production  class_SciTech  class_Special  class_Title  class_Writing  canon_ACTOR_IN_A_LEADING_ROLE  canon_ACTOR_IN_A_SUPPORTING_ROLE  canon_ACTRESS_IN_A_LEADING_ROLE  canon_ACTRESS_IN_A_SUPPORTING_ROLE  canon_ART_DIRECTION  canon_BEST_PICTURE  canon_CINEMATOGRAPHY  canon_DIRECTING  canon_DOCUMENTARY_Feature_  canon_DOCUMENTARY_Short_Subject_  canon_FILM_EDITING  canon_INTERNATIONAL_FEATURE_FILM  canon_MUSIC_Original_Score_  canon_MUSIC_Original_Song_  canon_Other  canon_SCIENTIFIC_AND_TECHNICAL_AWARD_Technical_Achievement_Award_  canon_SHORT_FILM_Animated_  canon_SHORT_FILM_Live_Action_  canon_SOUND_MIXING  canon_WRITING_Adapted_Screenplay_  canon_WRITING_Original_Sc